In [1]:
from IPython.core.magic import register_cell_magic, register_line_magic
import subprocess
import tempfile
import os
import re
import shutil

_JAVA_COMPILE_DIR = None
_JAVA_CLASSPATH = []


def _get_compile_dir():
    global _JAVA_COMPILE_DIR
    if _JAVA_COMPILE_DIR is None:
        _JAVA_COMPILE_DIR = tempfile.mkdtemp(prefix="java_magic_")
    return _JAVA_COMPILE_DIR


def _cleanup():
    global _JAVA_COMPILE_DIR, _JAVA_CLASSPATH
    if _JAVA_COMPILE_DIR and os.path.exists(_JAVA_COMPILE_DIR):
        shutil.rmtree(_JAVA_COMPILE_DIR, ignore_errors=True)
    _JAVA_COMPILE_DIR = None
    _JAVA_CLASSPATH = []


@register_line_magic
def javacp(line):
    global _JAVA_CLASSPATH
    path = line.strip().strip('"').strip("'")
    if os.path.exists(path):
        _JAVA_CLASSPATH.append(path)
        print("[OK] Added to classpath: " + path)
    else:
        print("[ERR] Not found: " + path)


@register_line_magic
def javacls(line):
    _cleanup()
    print("[OK] Java compile cache cleared")


@register_cell_magic
def java(line, cell):
    opts = []
    parts = line.strip().split()
    while parts and parts[0].startswith('--'):
        opts.append(parts.pop(0))
    
    force_new = '--new' in opts
    show_time = '--time' in opts
    
    if force_new:
        _cleanup()
    
    compile_dir = _get_compile_dir()
    
    public_class = re.search(r'public\s+class\s+(\w+)', cell)
    package_class = re.search(r'class\s+(\w+)', cell)
    
    if public_class:
        classname = public_class.group(1)
    elif package_class:
        classname = package_class.group(1)
    else:
        classname = "Main"
    
    if parts and parts[0][0].isupper() and not parts[0].isdigit():
        cmd_classname = parts.pop(0)
        if cmd_classname != classname and not public_class and not package_class:
            classname = cmd_classname
    else:
        cmd_classname = classname
    
    args = parts
    
    package_match = re.search(r'package\s+([\w.]+);', cell)
    if package_match:
        package_path = package_match.group(1).replace('.', os.sep)
        src_dir = os.path.join(compile_dir, package_path)
        os.makedirs(src_dir, exist_ok=True)
    else:
        src_dir = compile_dir
    
    filepath = os.path.join(src_dir, classname + ".java")
    
    code = cell
    if not re.search(r'class\s+\w+', cell):
        code = (
            "public class " + classname + " {\n"
            "    public static void main(String[] args) {\n"
            "        " + cell.replace('\n', '\n        ') + "\n"
            "    }\n"
            "}"
        )
    
    has_scanner = 'Scanner' in cell and 'System.in' in cell
    input_data = None
    
    if has_scanner and args:
        input_data = '\n'.join(args) + '\n'
        args = []
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(code)
    
    javac_cmd = ['javac', '-encoding', 'UTF-8']
    if _JAVA_CLASSPATH:
        cp = os.pathsep.join(_JAVA_CLASSPATH)
        javac_cmd.extend(['-cp', cp])
    javac_cmd.append(filepath)
    
    compile_result = subprocess.run(
        javac_cmd,
        capture_output=True,
        text=True,
        cwd=compile_dir
    )
    
    if compile_result.returncode != 0:
        errors = compile_result.stderr
        errors = errors.replace(compile_dir + os.sep, '')
        errors = errors.replace(compile_dir, '')
        print("[COMPILE ERROR]\n")
        print(errors)
        return
    
    java_cmd = ['java', '-Dfile.encoding=UTF-8']
    if _JAVA_CLASSPATH:
        cp = os.pathsep.join([compile_dir] + _JAVA_CLASSPATH)
        java_cmd.extend(['-cp', cp])
    else:
        java_cmd.extend(['-cp', compile_dir])
    
    if package_match:
        full_classname = package_match.group(1) + "." + cmd_classname
    else:
        full_classname = cmd_classname
    
    java_cmd.append(full_classname)
    java_cmd.extend(args)
    
    import time
    start = time.time()
    
    try:
        if input_data:
            run_result = subprocess.run(
                java_cmd,
                capture_output=True,
                text=True,
                cwd=compile_dir,
                input=input_data,
                timeout=10
            )
        else:
            run_result = subprocess.run(
                java_cmd,
                capture_output=True,
                text=True,
                cwd=compile_dir,
                timeout=10
            )
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start
        print("[TIMEOUT] Execution timed out after " + str(round(elapsed, 2)) + "s")
        if has_scanner and not input_data:
            print("\n[HINT] Scanner(System.in) detected but no input provided")
            print("       Solutions:")
            print("       1. Hardcode input values")
            print("       2. Pass input as args: %%java ClassName 20")
            print("       3. Use file redirection")
        return
    
    elapsed = time.time() - start
    
    if run_result.stdout:
        print(run_result.stdout, end='')
    
    if run_result.stderr:
        stderr = run_result.stderr
        harmless = ['Picked up JAVA_TOOL_OPTIONS', 'WARNING']
        if not any(h in stderr for h in harmless):
            print("\n[STDERR] " + stderr.strip())
    
    if show_time:
        print("\n[TIME] " + str(round(elapsed, 3)) + "s")


del java, javacp, javacls

练习：请用ArrayList存储学生姓名，删除长度小于3的名字

In [2]:
%%java Test01

import java.util.ArrayList;

public class Test01 {
    public static void main(String[] args) {
        ArrayList<String> list = new ArrayList<>();

        list.add("张三");
        list.add("李四");
        list.add("王五五");
        list.add("赵六六");

        for (int i = list.size() - 1; i >= 0; i--) {
            String name = list.get(i);

            if (name.length() < 3) {
                list.remove(i);
            }
        }

        System.out.println("过滤后:" + list);
    }
}

过滤后:[王五五, 赵六六]


LinkedList（底层是双向链表，增删快、查询慢）

In [3]:
%%java Test02

import java.util.LinkedList;

public class Test02 {
    public static void main(String[] args) {
        LinkedList<Integer> link = new LinkedList<>();

        link.add(10);
        link.add(20);

        link.addFirst(5);

        link.addLast(30);

        System.out.println(link);

        System.out.println("首元素:" + link.getFirst());

        System.out.println("尾元素:" + link.getLast());
    }
}

[5, 10, 20, 30]
首元素:5
尾元素:30


set(无序、不可重复、无索引)\
HashSet

In [4]:
%%java Test03

import java.util.HashSet;

public class Test03 {
    public static void main(String[] args) {
        HashSet<String> set = new HashSet<>();

        set.add("苹果");
        set.add("香蕉");
        set.add("橘子");
        set.add("苹果");

        for (String fruit : set) {
            System.out.println(fruit);
        }
    }
}

苹果
香蕉
橘子


需求：定义数组{10,20,10,30,20,40},使用set完成去重

In [5]:
%%java Test04

import java.util.HashSet;

public class Test04 {
    public static void main(String[] args) {
        int[] arr = {10, 20, 10, 30, 20, 40};

        HashSet<Integer> set = new HashSet<>();

        for (int num : arr) {
            set.add(num);
        }

        System.out.println("过滤后数据:" + set);
    }
}

过滤后数据:[20, 40, 10, 30]


Map双列集合(键值对 key-value，key必须唯一，value可以重复)\
存储，键不能重复，值可以重复；适合存储对象属性、表单数据

In [6]:
%%java Test05

import java.util.HashMap;

public class Test05 {
    public static void main(String[] args) {
        HashMap<Integer, String> map = new HashMap<>();

        map.put(1001, "张三");
        map.put(1002, "李四");
        map.put(1003, "王五");
        map.put(1001, "赵六");
        map.put(1004, "张三");

        System.out.println(map.get(1001));

        map.remove(1002);

        System.out.println(map.containsKey(1003));

        for (String value : map.values()) {
            System.out.println(value);
        }
    }
}

赵六
true
赵六
王五
张三


需求:HashMap统计字符串每个字符出现出现.

In [7]:
%%java Test06

import java.util.HashMap;

public class Test06 {
    public static void main(String[] args) {
        String str = "aabbcccdddd";

        HashMap<Character, Integer> countMap = new HashMap<>();

        for (int i = 0; i < str.length(); i++) {
            char c = str.charAt(i);

            if (countMap.containsKey(c)) {
                Integer value = countMap.get(c);
                value++;
                countMap.put(c, value);
            } else {
                countMap.put(c, 1);
            }
        }
        System.out.println("字符统计:" + countMap);
    }
}

字符统计:{a=2, b=2, c=3, d=4}


面向对象\
类：模板、描述一类事物共同的属性和行为\
对象：类的实例，根据类创建出来的个体，拥有类的属性和行为\
类是抽象的，对象是具体的，一个类可以有N个对象

In [8]:
%%java Test07

class Student {
    String no;
    int age;
    String name;
    String tel;

    public void study() {
        System.out.println(name + "正在学习，今年" + age + "岁");
    }

    public void play() {
        System.out.println(name + "正在玩耍");
    }
}

public class Test07 {
    public static void main(String[] args) {
        Student stu = new Student();
        stu.name = "张三";
        stu.age = 20;
        stu.no = "20231204090";
        stu.tel = "13800138000";

        stu.study();
        stu.play();
    }
}

张三正在学习，今年20岁
张三正在玩耍


类名  对象名 = new 类名();

In [9]:
%%java Test07

public class Test07 {
    public static void main(String[] args) {
        Student stu1 = new Student();
        Student stu2 = new Student();

        System.out.println("stu1:" + stu1);
        System.out.println("stu2:" + stu2);

        stu1.no = "001";
        stu1.age = 18;
        stu1.name = "张三";
        stu1.tel = "12345678901";

        stu1.study();
        stu1.play();

        stu2.study();
        stu2.play();
    }
}

stu1:Student@2ff4acd0
stu2:Student@38af3868
张三正在学习，今年18岁
张三正在玩耍
null正在学习，今年0岁
null正在玩耍


需求：定义手机类，包含品牌、价格属性、打电话、发短信方法，创建2个手机对象调用方法

In [10]:
%%java Test08

class Phone {
    String brand;
    double price;

    public void call(String number) {
        System.out.println(brand + "的手机,价格:" + price + "元，正在拨打:" + number);
    }

    public void sendSMS(String number, String content) {
        System.out.println(brand + "的手机,价格:" + price + "元，正在给" + number + "发送短信:" + content);
    }
}

public class Test08 {
    public static void main(String[] args) {
        Phone phone = new Phone();
        phone.brand = "华为";
        phone.price = 5999.0;

        phone.call("13800138000");
        phone.sendSMS("13800138000", "你好，在吗？");
    }
}

华为的手机,价格:5999.0元，正在拨打:13800138000
华为的手机,价格:5999.0元，正在给13800138000发送短信:你好，在吗？


需求：定义一个宠物类，私有属性name、weight;set方法限制体重>0,get方法获取属性

In [12]:
%%java Test09

class Person {
    private String name;
    private int age;

    public String getName() {
        return name;
    }

    public void setName(String name) {
        this.name = name;
    }

    public int getAge() {
        return age;
    }

    public void setAge(int age) {
        this.age = age;
    }
}

public class Test09 {
    public static void main(String[] args) {
        Person p = new Person();

        p.setName("张三");
        p.setAge(20);

        System.out.println(p.getName() + "的年龄:" + p.getAge());
    }
}

张三的年龄:20


In [ ]:
封装\
隐藏内部细节，对外提供安全访问入口\

实现步骤：\
使用private修饰成员变量，私有化，外部无法直接访问\
提供公共的getXXX（）获取值，setXXX（）设置值\
在公共接口中添加逻辑判断

构造方法：创建对象时new自动调用的方法，专门用于给对象属性初始化

特征：\
方法名与类名完全一致\
无返回值，不能void\
一个类默认自带无参构造方法；自定义有参构造方法之后，默认无参构造方法就会消失

需求：汽车类，私有品牌、颜色；提供无参、有参构造方法，输出对象信息